<h1>Libraries</h1>

In [ ]:
import dask.dataframe as dd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import gc

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from dask_ml.cluster import KMeans
from dask_ml.preprocessing import StandardScaler

from sklearn.cluster import MiniBatchKMeans

<h1>Part 1</h1>

<h3>Load dataset and categorize columns</h3>

In [ ]:
df = dd.read_csv("data.csv")

categorical_columns = [
    "Label",
    "Traffic Type",
    "Traffic Subtype",
    "Protocol",
]

# List of columns to exclude
unnecessary_columns = [
    "Flow ID", "Timestamp", "Src IP", "Dst IP", "Src Port", "Dst Port"
]

binary_columns = [

    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
]

# Filter out the unnecessary ones
not_numerical_columns = categorical_columns + unnecessary_columns + binary_columns
numerical_columns = [col for col in df.columns.tolist() if col not in not_numerical_columns]
num_df = df[numerical_columns]

<h3>Compute Basic Stats</h3>

In [ ]:
stats = num_df.describe().compute()

In [ ]:
# correlation = num_df.corr().compute()

Fallback if correlation variable is lost

In [ ]:
correlation = dd.read_csv("csv/correlation.csv").compute() 

# Get column names
col_names = correlation.columns.tolist()

# Create a new column with the column names as rows
correlation.insert(0, '', col_names)
correlation.set_index('', inplace=True)

In [ ]:
stats_correlation = stats.T.corr()

<h3>Print Basic Stats</h3>

In [ ]:
print(stats)
stats.to_csv("csv/stats.csv")

<h3>Plot Basic Stats</h3>

In [ ]:
for idx, row in stats.iterrows():
    if idx == "count":
        continue
    plt.figure()
    row.plot(figsize=(20,10), kind='bar')

    plt.title(idx)
    plt.xlabel("Columns")
    plt.ylabel("Values")
    plt.tight_layout()
    plt.show()

<h2>Correlation between columns</h2>

<h3>Heatmap</h3>

In [ ]:
# Mask upper triangle
mask = np.triu(np.ones_like(correlation, dtype=bool))

plt.figure(figsize=(20, 18))  # Increase size
sns.heatmap(correlation, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, annot=False, cbar_kws={"shrink": 0.5})
plt.title('Correlation Heatmap (Masked Upper Triangle)')
plt.show()

<h3>Print highest correlations</h3>

In [ ]:
high_corr_pairs = correlation.where(mask).stack()  # This converts the DataFrame to a Series with MultiIndex
high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.75]  # Filter the correlations greater than 0.75

# Print the results
print("Correlation pairs")
for idx, value in high_corr_pairs.sort_values(ascending=False).items():
    if idx[0] == idx[1]:
        # Skip self-correlation
        continue
    
    print(f"{value:.2f} {idx[0]} - {idx[1]}")

<h2>Statistical Correlation</h2>

<h3>Heatmap</h3>

In [ ]:
# Mask upper triangle
mask = np.triu(np.ones_like(stats_correlation, dtype=bool))

plt.figure(figsize=(20, 18))  # Increase size
sns.heatmap(stats_correlation, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, annot=False, cbar_kws={"shrink": 0.5})
plt.title('Correlation Heatmap (Masked Upper Triangle)')
plt.show()

<h3>Print</h3>

In [ ]:
print("Statistical Correlation pairs")
for idx, value in high_corr_pairs.sort_values(ascending=False).items():
    if idx[0] == idx[1]:
        # Skip self-correlation
        continue
    print(f"{value:.2f} {idx[0]} - {idx[1]}")

# Part 2

## Remove columns based on stats

### Get low variance features

In [ ]:
low_variance_features = stats.T[stats.T['std'] < 1e-6].index.tolist()
print("Low variance features:")
for feature in low_variance_features:
    print(feature)

### Drop features that include "Min"/"Max" and have high correlation with one that has "Mean" 

In [ ]:
col_to_drop = []
to_remove = ["Min", "Max", "Avg"]
to_keep = ["Mean", "Std"]
for idx, value in high_corr_pairs.sort_values(ascending=False).items():
    if idx[0] == idx[1]:
        continue
    
    if any(rem in idx[0] for rem in to_remove) and any(keep in idx[1] for keep in to_keep):
        col_to_drop.append(idx[0])
    
    if any(rem in idx[1] for rem in to_remove) and any(keep in idx[0] for keep in to_keep):
        col_to_drop.append(idx[1])

In [ ]:
col_to_drop.extend(low_variance_features)
col_to_drop.extend(unnecessary_columns)

col_to_drop = list(set(col_to_drop))

df_less_cols = df.drop(columns=col_to_drop)

In [ ]:
# Συνενώσεις/συνδυασμοί χαρακτηριστικών

# Λόγος Fwd/Bwd Packet Count
df_less_cols["Fwd_Bwd_Packet_Ratio"] = df["Total Fwd Packet"] / (df["Total Bwd packets"] + 1e-6)

# Λόγος συνολικού μήκους δεδομένων
df_less_cols["Fwd_Bwd_Length_Ratio"] = df["Total Length of Fwd Packet"] / (df["Total Length of Bwd Packet"] + 1e-6)

# Λόγος μέσου μεγέθους πακέτου Fwd/Bwd
df_less_cols["Fwd_Bwd_PacketLengthMean_Ratio"] = df["Fwd Packet Length Mean"] / (df["Bwd Packet Length Mean"] + 1e-6)

# Μέση τιμή ρυθμού δεδομένων (bytes/s)
df_less_cols["Avg_Bytes_per_s"] = (df["Flow Bytes/s"] + df["Fwd Packets/s"] + df["Bwd Packets/s"]) / 3

# Λόγος ενεργής/ανενεργής περιόδου
df_less_cols["Active_Idle_Ratio"] = df["Active Mean"] / (df["Idle Mean"] + 1e-6)

# Λόγος αρχικού παραθύρου FWD/BWD
df_less_cols["Win_Init_Ratio"] = df["FWD Init Win Bytes"] / (df["Bwd Init Win Bytes"] + 1e-6)




columns_to_drop = [
    "Total Fwd Packet", "Total Bwd packets",
    "Total Length of Fwd Packet", "Total Length of Bwd Packet",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "Flow Bytes/s", "Fwd Packets/s", "Bwd Packets/s",
    "Active Mean", "Idle Mean",
    "FWD Init Win Bytes", "Bwd Init Win Bytes"
]

# Αφαίρεση των στηλών από το DataFrame
df_less_cols = df_less_cols.drop(columns=columns_to_drop)


In [ ]:
dd.to_csv(df_less_cols, "less_cols_kmeans/less_cols*.csv")
# Clear memory
del df_less_cols
gc.collect()

In [ ]:
del df, num_df, stats, correlation, stats_correlation
gc.collect()

## Sampling

In [ ]:
sampled_df = df_less_cols.sample(frac=0.1, random_state=42)
dd.to_csv(sampled_df, 'sampled/sampled*.csv')
del sampled_df
gc.collect()

## Clustering

### K-Means

In [ ]:
df_less_cols = dd.read_csv("less_cols_kmeans/less_cols*.csv")
print(df_less_cols.columns)

In [ ]:
# df_less_cols = dd.read_csv("sampled/sampled*.csv")
print(df_less_cols.columns)
# Επιλογή χαρακτηριστικών για clustering

selected_features = [
    'Flow Duration',
    'Flow IAT Mean',
    'Fwd IAT Mean',
    'Bwd IAT Mean',
    'Packet Length Mean',
    'Average Packet Size',
    'Down/Up Ratio',
    'Subflow Fwd Packets',
    'Subflow Bwd Packets',
    'Fwd_Bwd_Packet_Ratio',
    'Avg_Bytes_per_s',
    'Active_Idle_Ratio'
]

features = df_less_cols[selected_features]

# Κλιμάκωση (scaling) των χαρακτηριστικών
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features.compute())
del features
gc.collect()

#### Dask K-Means

In [ ]:
# Ορισμός και εκπαίδευση του KMeans
kmeans = KMeans(n_clusters=3, init_max_iter=10)
kmeans.fit(X_scaled)

In [ ]:
del X_scaled
gc.collect()

In [ ]:
# Πρόβλεψη ετικετών (labels)
labels = kmeans.labels_
print("Labels:", labels)

In [ ]:
# Προσθήκη των ετικετών στο αρχικό Dask DataFrame
df_less_cols = df_less_cols.compute()  # Μετατροπή σε Pandas DataFrame για ευκολότερη επεξεργασία
df_less_cols['Cluster'] = labels

# Αποθήκευση ή υλοποίηση υπολογισμών
result = df_less_cols
print(result["Cluster"].head())

In [ ]:
# Convert to Dask DataFrame
labels_df = labels.to_dask_dataframe(columns=["Cluster"])

# Repartition labels to match df_less_cols
labels_df = labels_df.repartition(npartitions=df_less_cols.npartitions)

# Concatenate safely
df_clustered = dd.concat([df_less_cols.reset_index(drop=True), labels_df.reset_index(drop=True)]).compute()

#### MiniBatchKMeans

In [ ]:
# print(X_scaled.compute().std(axis=0))  # Should not be all near 0
X_df = X_scaled.compute()
print(X_df.head())
print(X_df.std())
print(X_df.nunique())

In [ ]:
kmeans = MiniBatchKMeans(n_clusters=10, batch_size=1000, random_state=0, verbose=1).fit(X_scaled)

#### Elbow method using MiniBatchKMeans

In [ ]:

inertia = []
for k in range(2, 11):
    print(f"k: {k}")
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=1000, random_state=0, verbose=1).fit(X_scaled)
    inertia.append(kmeans.inertia_)


#### Plot

In [ ]:
plt.plot(range(2, 11), inertia, marker='o')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

#### Get clustering info

In [ ]:
print(kmeans.labels_)
labels = kmeans.labels_
print(labels)
import pandas as pd

labels_dd = dd.from_pandas(pd.DataFrame(labels, columns=["cluster"]))
# df_sampled_result = dd.concat([df_less_cols, labels_dd]).sample(frac=0.01).compute()

result = dd.concat([df_less_cols, labels_dd])
result.to_csv('output/clustered_data_*.csv', index=False, single_file=False).compute()


### DBSCAN

### Preprocessing

In [ ]:
# Preprocessing pipelines
standard_scaler = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(transformers=[
    ('num', standard_scaler, numerical_columns),
    ('cat', categorical_transformer, categorical_columns),
], remainder='passthrough')  # leaves binary columns as they are
